This notebook explores the new 2022, 2023 and 2024 data files and compares them to the original dataset. The goal is to identify any differences in columns, site information, and data values. They will be combined into a single dataset.

In [148]:
import pandas as pd
from wrwc.config import RAW_DATA_DIR, PROCESSED_DATA_DIR

In [149]:
df_2022 = pd.read_excel(RAW_DATA_DIR / 'WRWC 2023-2022.xlsx', sheet_name='2022')
df_2023 = pd.read_excel(RAW_DATA_DIR / 'WRWC 2023-2022.xlsx', sheet_name='2023')
df_2024 = pd.read_excel(RAW_DATA_DIR / 'WRWC_2024.xlsx', sheet_name='WRWC')
df_orig = pd.read_csv(RAW_DATA_DIR / 'WoonasquatucketData.csv', parse_dates=['Date of Sample'])
# df_orig = pd.read_csv(PROCESSED_DATA_DIR / 'wrwc-processed-data-20260714.csv')

In [150]:
def compare_lists(l1, l2):
    not_in_l1 = [x for x in l2 if x not in l1]
    not_in_l2 = [x for x in l1 if x not in l2]

    return not_in_l1, not_in_l2


In [151]:
r1, r2 = compare_lists(df_2022.columns, df_2023.columns)

print("Columns in 2023 but not in 2022:", r1)
print("Columns in 2022 but not in 2023:", r2)

Columns in 2023 but not in 2022: ['Station Name', 'source_name']
Columns in 2022 but not in 2023: ['WW ID']


Change "WW ID" to "Station Name" in 2022

In [152]:
r1, r2 = compare_lists(df_2023.columns, df_2024.columns)

print("Columns in 2024 but not in 2023:", r1)
print("Columns in 2023 but not in 2024:", r2)

Columns in 2024 but not in 2023: []
Columns in 2023 but not in 2024: ['source_name']


In [153]:
df_2022_edited = df_2022.rename(columns={"WW ID": "Station Name"})
df_combined = pd.concat([df_2022_edited, df_2023, df_2024], ignore_index=True)

In [154]:
# Standardize to match other data file
df_combined_edited = (
    df_combined
    .rename(columns={'Station Name': 'WW ID', 'source_name': 'Source.Name'})
    .drop(columns=['MONITOR 1', 'MONITOR 2'])
    .assign(**{'Watershed Code': 'WO'})
)
# Reorder to match original dataset
df_combined_edited = df_combined_edited[df_orig.columns]

In [155]:
r1, r2 = compare_lists(df_orig.columns, df_combined_edited.columns)
print("Columns in new but not in orig:", r1)
print("Columns in orig but not in new:", r2)

Columns in new but not in orig: []
Columns in orig but not in new: []


In [156]:
df_combined_edited.head()


,Source.Name,WW ID,Date of Sample,Time,Sample Type,Sample Media,Depth,Parameter,Concentration,Unit,...,Lab Name,Analytical Method Number,Sediment Particle Size,Particle Size Unit,Fish Sample Type,Fish Taxa,Comments,Monitoring location,Watershed,Watershed Code
0,NaN,WW227,2022-05-07,07:00:00,Grab,Water,0.2,Chloride - 00940,83.000,mg/l,...,URIWW,325_M(A): Chloride in Water by Colorimetry,NaN,NaN,NaN,NaN,NaN,Woonasquatucket River @ Donigian Park,Woonasquatucket River,WO
1,NaN,WW227,2022-05-07,07:00:00,Grab,Water,0.2,Enterococci - 31639,20.000,MPN/100,...,URIWW,NaN,NaN,NaN,NaN,NaN,NaN,Woonasquatucket River @ Donigian Park,Woonasquatucket River,WO
2,NaN,WW227,2022-05-07,07:00:00,Grab,Water,0.2,"Nitrate + Nitrite, Dissolved - 00631",0.585,mg/l,...,URIWW,4500-NO3(F): Nitrate in Water- Automated Cadmi...,NaN,NaN,NaN,NaN,NaN,Woonasquatucket River @ Donigian Park,Woonasquatucket River,WO
3,NaN,WW227,2022-05-07,07:00:00,Grab,Water,0.2,"Nitrogen, Ammonia Dissolved as N - 00608",0.070,mg/l,...,URIWW,4500-NH3(G): Ammonia in Water Using Automated ...,NaN,NaN,NaN,NaN,NaN,Woonasquatucket River @ Donigian Park,Woonasquatucket River,WO
4,NaN,WW227,2022-05-07,07:00:00,Grab,Water,0.2,"Nitrogen, Total (unfiltered) - 00600",0.743,mg/l,...,URIWW,NaN,NaN,NaN,NaN,NaN,NaN,Woonasquatucket River @ Donigian Park,Woonasquatucket River,WO


In [157]:
# Compare sites
df_combined_edited['WW ID'].unique()

array(['WW227', 'WW308', 'WW226', 'WW635', 'WW437', 'WW657', 'WW659'],
      dtype=object)

In [158]:
df_orig['WW ID'].unique()

array(['WW016', 'WW024', 'WW046', 'WW052', 'WW061', 'WW065', 'WW113',
       'WW114', 'WW123', 'WW124', 'WW125', 'WW126', 'WW144', 'WW153',
       'WW201', 'WW226', 'WW227', 'WW238', 'WW239', 'WW240', 'WW241',
       'WW308', 'WW437', 'WW508', 'WW518', 'WW635', 'WW679', 'WW680'],
      dtype=object)

There are two new sites: 'WW657', 'WW659'. Their information is found in the 2024 file. I manually added them to the SiteInfo.csv file.

In [159]:
# Combine datasets
df_out = pd.concat([df_orig, df_combined_edited], ignore_index=True)
df_out.to_csv(RAW_DATA_DIR / 'WoonasquatucketData_through_2024.csv', index=False)

In [160]:
df_out['Date of Sample'].tail()

26885   2024-10-26
26886   2024-10-26
26887   2024-10-26
26888   2024-10-26
26889   2024-10-26
Name: Date of Sample, dtype: datetime64[ns]